In [1]:
# !pip install --upgrade tensorflow-model-optimization
# !pip uninstall -y numpy
# !pip install numpy==1.26.4
# import os
# os.kill(os.getpid(), 9)

In [2]:
import tensorflow as tf
from tensorflow import keras as kr
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import tensorflow_model_optimization as mo
import shutil

In [3]:
(x_train, y_train) , (x_test, y_test) = kr.datasets.mnist.load_data()
x_train = x_train / 255
x_test = x_test / 255
x_train_flattened = x_train.reshape(len(x_train), 28*28)
x_test_flattened = x_test.reshape(len(x_test), 28*28)

11490434/11490434 [==============================] - 0s 0us/step


In [4]:
model = kr.Sequential([
    kr.layers.Flatten(input_shape=(28, 28)),
    kr.layers.Dense(100, activation='relu'),
    kr.layers.Dense(10, activation='sigmoid')
])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.fit(x_train, y_train, epochs=5)

Epoch 1/5
1875/1875 [==============================] - 5s 2ms/step - loss: 0.2805 - accuracy: 0.9207
Epoch 2/5
1875/1875 [==============================] - 9s 5ms/step - loss: 0.1270 - accuracy: 0.9624
Epoch 3/5
1875/1875 [==============================] - 10s 5ms/step - loss: 0.0895 - accuracy: 0.9734
Epoch 4/5
1875/1875 [==============================] - 6s 3ms/step - loss: 0.0673 - accuracy: 0.9797
Epoch 5/5
1875/1875 [==============================] - 4s 2ms/step - loss: 0.0536 - accuracy: 0.9836


In [5]:
model.evaluate(x_test,y_test)

313/313 [==============================] - 1s 2ms/step - loss: 0.0816 - accuracy: 0.9738


[0.08155198395252228, 0.973800003528595]

In [6]:
model.save("./2) tensorflow.model.save")
shutil.make_archive('2) tensorflow.model.save', 'zip', '.')

'/content/2) tensorflow.model.save.zip'

In [12]:
# Post Training Quantization without Quantization
converter = tf.lite.TFLiteConverter.from_saved_model("./2) tensorflow.model.save")
tflite_model = converter.convert()

In [10]:
# Post Training Quantization with Quantization
converter = tf.lite.TFLiteConverter.from_saved_model("./2) tensorflow.model.save")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

In [13]:
print(len(tflite_model),len(tflite_quant_model))

319932 85992


In [14]:
with open("3) tflite_model.tflite", "wb") as f:
    f.write(tflite_model)

In [15]:
with open("4) tflite_quant_model.tflite", "wb") as f:
    f.write(tflite_quant_model)

In [16]:
# Quantization Aware Training
quantize_model = mo.quantization.keras.quantize_model
qa_model = quantize_model(model)
qa_model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
qa_model.summary()
qa_model.fit(x_train, y_train, epochs=1)
qa_model.evaluate(x_test, y_test)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer (QuantizeLa  (None, 28, 28)            3         
 yer)                                                            
                                                                 
 quant_flatten (QuantizeWra  (None, 784)               1         
 pperV2)                                                         
                                                                 
 quant_dense (QuantizeWrapp  (None, 100)               78505     
 erV2)                                                           
                                                                 
 quant_dense_1 (QuantizeWra  (None, 10)                1015      
 pperV2)                                                         
                                                                 
Total params: 79524 (310.64 KB)
Trainable params: 79510 

[0.07721590250730515, 0.9771999716758728]

In [17]:
converter = tf.lite.TFLiteConverter.from_keras_model(qa_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_qa_model = converter.convert()

/usr/local/lib/python3.11/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [18]:
len(tflite_qa_model)

82704

In [19]:
with open("5) tflite_qaware_model.tflite", 'wb') as f:
    f.write(tflite_qa_model)